<a href="https://www.kaggle.com/code/avikdas567/image2biomass-prediction?scriptVersionId=291647864" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/csiro-biomass/train/ID853954911.jpg
/kaggle/input/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/csiro-biomass/train/ID384648061.jpg
/kaggle/input/csiro-biomass/train/ID1563418511.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID482555369.jpg
/kaggle/input/csiro-biomass/train/ID638711343.jpg
/kaggle/input/c

In [2]:
import os
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.multioutput import MultiOutputRegressor

# ====================================================
# CONFIGURATION & PATHS
# ====================================================
class CFG:
    seed = 42
    # The root directory where the competition data lives
    DATA_ROOT = "/kaggle/input/csiro-biomass"
    
    # Specific file paths
    TRAIN_CSV = os.path.join(DATA_ROOT, "train.csv")
    TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
    SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.csv")
    
    # Target columns to predict
    target_cols = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']

def seed_everything(seed):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(CFG.seed)

# ====================================================
# FEATURE EXTRACTION
# ====================================================
def extract_handcrafted_features(image_path):
    """
    Extracts vegetation metrics (Greenness, Texture, Brightness) using standard OpenCV.
    Does NOT require a GPU or Internet.
    """
    try:
        # Load Image
        img = cv2.imread(image_path)
        if img is None:
            # Fallback if image is missing/corrupt
            return np.zeros(14) 
        
        # Convert color spaces
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) # Hue, Saturation, Value
        
        # 1. Basic Color Stats (RGB)
        mean_rgb = np.mean(img_rgb, axis=(0,1))
        std_rgb = np.std(img_rgb, axis=(0,1))
        
        # 2. Basic Color Stats (HSV - captures brightness & "colorfulness")
        mean_hsv = np.mean(img_hsv, axis=(0,1))
        std_hsv = np.std(img_hsv, axis=(0,1))
        
        # 3. Vegetation Indices
        # Cast to float to safely do math
        R = img_rgb[:,:,0].astype(float)
        G = img_rgb[:,:,1].astype(float)
        B = img_rgb[:,:,2].astype(float)
        
        # Excess Green (ExG): Standard metric for green vegetation
        # Formula: 2*G - R - B
        exg = 2*G - R - B
        mean_exg = np.mean(exg)
        
        # Green-Red Difference
        # Formula: (G - R) / (G + R)
        denom = G + R + 1e-6 # Avoid divide by zero
        grvi = (G - R) / denom
        mean_grvi = np.mean(grvi)
        
        # Combine all features
        features = np.concatenate([
            mean_rgb, std_rgb, 
            mean_hsv, std_hsv, 
            [mean_exg, mean_grvi]
        ])
        
        return features
        
    except Exception as e:
        return np.zeros(14)

# ====================================================
# DATA PROCESSING PIPELINE
# ====================================================
def run_pipeline():
    print("1. Loading Data...")
    train_df = pd.read_csv(CFG.TRAIN_CSV)
    test_df = pd.read_csv(CFG.TEST_CSV)
    sample_sub = pd.read_csv(CFG.SAMPLE_SUB)
    
    # Fix Image Paths: Join Root + Relative Path (e.g., "train/ID123.jpg")
    train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(CFG.DATA_ROOT, x))
    test_df['full_path'] = test_df['image_path'].apply(lambda x: os.path.join(CFG.DATA_ROOT, x))

    # --- Prepare Training Data ---
    # The data is "Long" (5 rows per image). We pivot it to "Wide" (1 row per image).
    train_df['base_id'] = train_df['sample_id'].apply(lambda x: x.split('__')[0])
    
    pivot_train = train_df.pivot_table(
        index=['base_id', 'full_path'],
        columns='target_name',
        values='target'
    ).reset_index()
    
    # Ensure columns correspond exactly to our target list
    pivot_train = pivot_train[['base_id', 'full_path'] + CFG.target_cols]
    
    # --- Prepare Test Data ---
    test_df['base_id'] = test_df['sample_id'].apply(lambda x: x.split('__')[0])
    # We only need one row per image to run extraction
    unique_test = test_df.drop_duplicates(subset=['base_id'])[['base_id', 'full_path']].reset_index(drop=True)

    print(f"   Training on {len(pivot_train)} images.")
    print(f"   Testing on {len(unique_test)} images.")

    # --- Feature Extraction ---
    print("\n2. Extracting Features (Train)...")
    X_train = []
    # Using tqdm to show a progress bar
    for path in tqdm(pivot_train['full_path']):
        X_train.append(extract_handcrafted_features(path))
    X_train = np.array(X_train)
    
    print("   Extracting Features (Test)...")
    X_test = []
    for path in tqdm(unique_test['full_path']):
        X_test.append(extract_handcrafted_features(path))
    X_test = np.array(X_test)
    
    y_train = pivot_train[CFG.target_cols].values
    
    # --- Model Training ---
    print("\n3. Training Model (Ridge Regression)...")
    # Pipeline: Normalize Data -> Train Ridge
    model = make_pipeline(
        StandardScaler(),
        MultiOutputRegressor(Ridge(alpha=1.0, random_state=CFG.seed))
    )
    
    # Handle any NaNs from bad images by filling with 0
    X_train = np.nan_to_num(X_train)
    X_test = np.nan_to_num(X_test)
    
    model.fit(X_train, y_train)
    print("   Model trained successfully.")
    
    # --- Prediction ---
    print("\n4. Predicting...")
    y_pred = model.predict(X_test)
    
    # Biomass cannot be negative, so clip at 0
    y_pred = np.maximum(y_pred, 0)
    
    # --- Submission ---
    print("5. Creating Submission File...")
    
    # Create a lookup dictionary: base_id -> {TargetName: Value}
    pred_map = {}
    for i, base_id in enumerate(unique_test['base_id'].values):
        preds = y_pred[i]
        pred_map[base_id] = {
            'Dry_Green_g': preds[0],
            'Dry_Dead_g': preds[1],
            'Dry_Clover_g': preds[2],
            'GDM_g': preds[3],
            'Dry_Total_g': preds[4]
        }
        
    # Map predictions to the official sample_submission format
    final_targets = []
    for idx, row in sample_sub.iterrows():
        sample_id = row['sample_id']
        base_id = sample_id.split('__')[0]
        target_name = sample_id.split('__')[1]
        
        if base_id in pred_map:
            val = pred_map[base_id].get(target_name, 0)
        else:
            val = 0 # Safety fallback
            
        final_targets.append(val)
        
    sample_sub['target'] = final_targets
    
    # Save
    sample_sub.to_csv('submission.csv', index=False)
    print("   Saved 'submission.csv'. Ready to submit!")
    print(sample_sub.head())

if __name__ == '__main__':
    run_pipeline()

1. Loading Data...
   Training on 357 images.
   Testing on 1 images.

2. Extracting Features (Train)...


100%|██████████| 357/357 [03:34<00:00,  1.66it/s]


   Extracting Features (Test)...


100%|██████████| 1/1 [00:00<00:00,  1.60it/s]


3. Training Model (Ridge Regression)...
   Model trained successfully.

4. Predicting...
5. Creating Submission File...
   Saved 'submission.csv'. Ready to submit!
                    sample_id     target
0  ID1001187975__Dry_Clover_g   2.237666
1    ID1001187975__Dry_Dead_g  18.879808
2   ID1001187975__Dry_Green_g  33.198538
3   ID1001187975__Dry_Total_g  54.315857
4         ID1001187975__GDM_g  35.436206
